<a href="https://colab.research.google.com/github/SohaHussain/CS336-Language-Modelling-from-scratch/blob/main/CS336_Lecture_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Custom Model

In [2]:
import torch
from torch import nn
import numpy as np

In [3]:
class Linear(nn.Module):
    """Simple linear layer."""
    def __init__(self, input_dim: int, output_dim: int):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(input_dim, output_dim) / np.sqrt(input_dim)) # to rescale

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x @ self.weight

In [4]:
class LinearModel(nn.Module):
    def __init__(self, dim: int, num_layers: int):
        super().__init__()
        self.layers = nn.ModuleList([
            Linear(dim, dim)
            for i in range(num_layers)
        ])
        self.final = Linear(dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Apply linear layers
        B, D = x.size()
        for layer in self.layers:
            x = layer(x)

        # Apply final head
        x = self.final(x)
        assert x.size() == torch.Size([B, 1])

        # Remove the last dimension
        x = x.squeeze(-1)
        assert x.size() == torch.Size([B])

        return x

In [5]:
def get_num_parameters(model: nn.Module) -> int:
    return sum(param.numel() for param in model.parameters())

In [6]:
def custom_model():

    D = 64  # Dimension
    num_layers = 2
    model = LinearModel(dim=D, num_layers=num_layers)

    param_sizes = [
        (name, param.numel())
        for name, param in model.state_dict().items()
    ]
    assert param_sizes == [
        ("layers.0.weight", D * D),
        ("layers.1.weight", D * D),
        ("final.weight", D),
    ]
    num_parameters = get_num_parameters(model)
    assert num_parameters == (D * D) + (D * D) + D

    B = 8  # Batch size
    x = torch.randn(B, D)
    y = model(x)
    assert y.size() == torch.Size([B])